# Comprehensive Dynamical Analysis for AKOrN Models

This notebook performs comprehensive dynamical analysis on all 12 cases (sweep_20250714_601485.opbs_0 through ..._601496.opbs_11) from the parameter sweep using the `AKOrNDynamicalAnalyzer` class. 

This set of data is a result where 
- T of the 0'th layer is long (up to 127), and gamma is small (0.01).
- kernel sizes of the 0'th layer were smaller (3 or 5).

We analyze energy dynamics, temporal evolution, convergence properties, and network characteristics across all gamma and T value combinations.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import sys
import os
import json
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Add project root to path (since we're in notebooks/)
# project_root = Path.cwd().parent
# sys.path.insert(0, str(project_root))

# Setup imports
from source.models.classification.my_knet import MyAKOrN
from source.models.classification.analysis_utils import AKOrNDynamicalAnalyzer, AKOrNStaticAnalyzer
from source.models.classification.sweep_analysis_utils import (
    load_model_from_sweep,
    load_all_sweep_models, 
    create_train_loader,
    create_test_loader,
    plot_energy_dynamics
)
from source.kuramoto_network_metrics import (
    compute_all_metrics, 
    spectral_metrics, 
    strength_metrics,
    community_metrics,
    path_metrics,
    graph_from_K
)
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import networkx as nx

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cpu':
    #project_root = Path.cwd()
    project_root = Path.cwd().parent
elif device.type == 'cuda':
    project_root = Path.cwd()
print(f"Using device: {device}")
print(f"Project root: {project_root}")

## Load All 12 Sweep Cases

We load all models from sweep_20250714_*.opbs_0 through sweep_20250714_*.opbs_11 for comprehensive analysis.

In [ ]:
# Generate all 12 cases (opbs_0 through opbs_11)
all_cases = []
for i in range(12):
    all_cases.append({
        "dir": f"sweep_20250714_6014{85+i}.opbs_{i}",
        "index": i,
        "name": f"Case {i}"
    })

print(f"Generated {len(all_cases)} cases for comprehensive analysis:")
for case in all_cases:
    print(f"  {case['name']}: {case['dir']}")

In [ ]:
# Load all 11 models
loaded_models = {}
parameter_summary = []

for case in all_cases:
    result = load_model_from_sweep(case["dir"], project_root / "results", device)
    if result is not None:
        model, config = result
        loaded_models[case["name"]] = {
            "model": model,
            "config": config,
            "gamma": config["gamma"],
            "T": config["T"],
            "sweep_dir": case["dir"],
            "index": case["index"]
        }
        
        # Add to parameter summary
        parameter_summary.append({
            "case": case["name"],
            "index": case["index"],
            "gamma": config["gamma"],
            "T": config["T"],
            "ksizes": config["ksizes"],
            "sweep_dir": case["dir"]
        })

print(f"\nSuccessfully loaded {len(loaded_models)} models for comprehensive analysis")

# Create parameter summary DataFrame
param_df = pd.DataFrame(parameter_summary)
print("\nParameter Summary:")
print(param_df.to_string(index=False))

## Setup Test Data

Load test data for comprehensive analysis.

In [ ]:
# Load test data
test_loader = create_test_loader(batch_size=64, data_dir=str(project_root / 'data'))

# Get sample inputs for analysis
test_batch = next(iter(test_loader))
test_input, test_labels = test_batch
test_input = test_input.to(device)

# Use first sample for detailed analysis
sample_input = test_input[0:1]  # Single sample
sample_batch = test_input[0:8]  # Small batch for batch analysis

print(f"Sample input shape: {sample_input.shape}")
print(f"Sample batch shape: {sample_batch.shape}")
print(f"Sample label: {test_labels[0].item()}")

## Comprehensive Energy Dynamics Analysis

Analyze energy dynamics across all 12 models and multiple layers.

In [ ]:
# Extract energy dynamics for all models and layers
energy_dynamics_data = {}

for name, model_data in loaded_models.items():
    print(f"\nAnalyzing energy dynamics for {name}...")
    model = model_data["model"]
    gamma = model_data["gamma"]
    ksizes= model_data["config"]["ksizes"]
    T = model_data["T"]
    case_index = model_data["index"]
    
    layer_dynamics = {}
    
    # Analyze layers 0, 1, 2
    for layer_idx in range(3):
        try:
            analyzer = AKOrNDynamicalAnalyzer(model, layer_idx=layer_idx, device=device)
            energy_data = analyzer.extract_energy_dynamics(sample_input)
            
            # Check if energy_data contains data for this layer
            if energy_data and layer_idx in energy_data:
                trajectory = energy_data[layer_idx]['trajectory']
                layer_dynamics[layer_idx] = {
                    'trajectory': trajectory,
                    'final_energy': trajectory[-1],
                    'initial_energy': trajectory[0],
                    'energy_change': trajectory[-1] - trajectory[0],
                    'max_energy': max(trajectory),
                    'min_energy': min(trajectory)
                }
                print(f"  Layer {layer_idx}: Final energy = {trajectory[-1]:.4f}")
            else:
                print(f"  Layer {layer_idx}: No energy data available")
                
        except Exception as e:
            print(f"  Layer {layer_idx}: Error - {e}")
    
    energy_dynamics_data[name] = {
        'layers': layer_dynamics,
        'gamma': gamma,
        'T': T,
        'ksizes': ksizes,
        'case_index': case_index
    }

print(f"\nCompleted energy dynamics analysis for {len(energy_dynamics_data)} models")

In [ ]:
plot_energy_dynamics(dynamics_data=energy_dynamics_data,
        X_values=[[3,7,5],[5,7,5]],
        Y_values=[[3,3,3],[7,3,3],[15,3,3],[31,3,3],[63,3,3],[127,3,3]],
        X_label="ksizes",
        Y_label="T",
        data_type="Training",
        title="Training"
        )